# Lab14-1 - 文字分類模型比較


- 使用 Hugging Face `datasets` 載入 AG News 新聞分類資料集。
- 透過 `DatasetProcessor` 建立 tokenizer、vocab 與 padded batch。
- 比較 Transformer、RNN、LSTM 與 GRU 分類器。
- 透過 TensorBoard 記錄 loss 與 accuracy。


## 1. Import必要套件


In [8]:
%pip install datasets

import torch
import torch.nn as nn
from torch.utils.tensorboard import SummaryWriter
from tqdm import tqdm

from data_loader import DatasetProcessor
from network import TransformerClassifier, RNNClassifier, LSTMClassifier, GRUClassifier


Note: you may need to restart the kernel to use updated packages.


## 2. 訓練與驗證函式


In [ ]:
def create_mask(src):
    # Transformer 會在 src 前面加入 CLS token，因此 mask 也要補上同一個位置
    # True 代表該位置是 <pad>，CLS 與有效 token 都必須保留給 attention 使用
    cls_mask = torch.zeros((1, src.size(1)), dtype=torch.bool, device=src.device)
    pad_mask = src == 1
    return torch.cat([cls_mask, pad_mask], dim=0)

def train(model, data_loader, epoch, writer, optimizer=None, criterion=None, device=None, model_name=None):
    model.train()
    device = device or torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    criterion = criterion or nn.CrossEntropyLoss()
    # 第一次呼叫時建立 optimizer，之後沿用同一個 optimizer 保留 Adam 的動量狀態
    optimizer = optimizer or torch.optim.Adam(model.parameters(), lr=5e-4)
    loader = tqdm(data_loader.train_dataloader, total=data_loader.train_len, desc=f'Epoch {epoch}')

    total_loss = 0.0
    total = correct = 0
    for label, text, offsets in loader:
        label = label.to(device)
        text = text.to(device)
        optimizer.zero_grad()

        if model_name == 'transformer':
            # Transformer 使用 mask 避免 attention 到 <pad> token
            src_mask = create_mask(text)
            output = model(text, src_mask.to(device))
        else:
            output = model(text)

        loss = criterion(output, label)
        loss.backward()
        optimizer.step()

        # loss 乘上 batch size 後再累加，最後可得到整個 epoch 的平均 loss
        total_loss += loss.item() * label.size(0)
        correct += (output.argmax(1) == label).sum().item()
        total += label.size(0)
        loader.set_postfix(loss=loss.item())

    writer.add_scalar('Loss/train', total_loss / total, epoch)
    writer.add_scalar('Accuracy/train', correct / total * 100, epoch)
    return optimizer


def valid(model, data_loader, writer, epoch, criterion=None, device=None, model_name=None):
    model.eval()
    device = device or torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    criterion = criterion or nn.CrossEntropyLoss()
    total_loss = 0.0
    total = correct = 0
    loader = tqdm(data_loader.test_dataloader, total=data_loader.test_len, desc=f'Epoch {epoch}')

    for label, text, offsets in loader:
        label = label.to(device)
        text = text.to(device)

        # 驗證階段不更新權重，只計算 logits、loss 與 accuracy
        with torch.no_grad():
            if model_name == 'transformer':
                src_mask = create_mask(text)
                output = model(text, src_mask.to(device))
            else:
                output = model(text)

            loss = criterion(output, label)
            _, predicted = torch.max(output.data, 1)
            total += label.size(0)
            correct += (predicted == label).sum().item()
            total_loss += loss.item() * label.size(0)
            loader.set_postfix(loss=loss.item())

    writer.add_scalar('Loss/valid', total_loss / total, epoch)
    writer.add_scalar('Accuracy/valid', correct / total * 100, epoch)


## 3. 建立資料集與模型


In [10]:
batch_size = 256
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

# DatasetProcessor 會建立 AG News 的 vocab、padding 後的 batch 與 train/test dataloader
data_process = DatasetProcessor(batch_size, device)

# 四種模型共用相同 vocab、embedding 維度與分類數，讓後續比較聚焦在序列模型差異
vocab_size = len(data_process.vocab)
emb_size = 32
hidden_size = 128
nclass = 4
nlayers = 2

models = {
    'transformer': TransformerClassifier(vocab_size, emb_size, nhead=4, nhid=hidden_size, nlayers=nlayers, nclass=nclass),
    'rnn': RNNClassifier(vocab_size, emb_size, hidden_size, nclass, nlayers=nlayers),
    'lstm': LSTMClassifier(vocab_size, emb_size, hidden_size, nclass, nlayers=nlayers),
    'gru': GRUClassifier(vocab_size, emb_size, hidden_size, nclass, nlayers=nlayers),
}


Using device: cuda


/workspace/liangfu/Lab14-1/data_loader.py:65: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  token_to_idx = torch.load(cache_path)
/opt/conda/lib/python3.11/site-packages/tor

In [11]:
def count_parameters(model):
    # 同時計算總參數量與需要訓練的參數量，方便比較模型大小
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable


for network_name, model in models.items():
    total, trainable = count_parameters(model)
    print(f'{network_name:12s} total: {total:,} | trainable: {trainable:,}')


transformer  total: 2,106,692 | trainable: 2,106,692
rnn          total: 2,135,396 | trainable: 2,135,396
lstm         total: 2,296,676 | trainable: 2,296,676
gru          total: 2,242,916 | trainable: 2,242,916


## 4. 訓練模型


In [ ]:
num_epochs = 20

# 逐一訓練四種模型，並將各自的 loss 與 accuracy 寫到不同 TensorBoard run
for network_name, model in models.items():

    writer = SummaryWriter(f'runs/{network_name}')
    model = model.to(device)
    optimizer = None

    for epoch in range(num_epochs):
        optimizer = train(model, data_process, epoch, writer, device=device, optimizer=optimizer, model_name=network_name)
        valid(model, data_process, writer, epoch, device=device, model_name=network_name)

    writer.close()


## 5. TensorBoard


In [12]:
%load_ext tensorboard
%tensorboard --logdir runs


The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


Reusing TensorBoard on port 6008 (pid 2213878), started 0:09:21 ago. (Use '!kill 2213878' to kill it.)